# S0 - Kiểm toán dữ liệu VN30F1M và gộp 1m -> 5m
Cấu trúc trên Google Drive: `MyDrive/KE_Recovery/{data,src,results}`. Chỉ cần `data/ohlc_export.csv` và thư mục `src/ke_research`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/KE_Recovery'
import sys, os, json, subprocess
sys.path.insert(0, f'{ROOT}/src')
OUT = f'{ROOT}/results/S0'
os.makedirs(OUT, exist_ok=True)
from ke_research import data

In [ ]:
df1 = data.load_1m(f'{ROOT}/data/ohlc_export.csv')
audit = data.audit_1m(df1)
print(json.dumps(audit, indent=1, ensure_ascii=False))

In [ ]:
df5 = data.resample(df1, '5min')
checks = data.check_resample(df1, df5)
print(json.dumps(checks, indent=1))
assert all(v for k, v in checks.items() if k.endswith('_equal') or k == 'minutes_conserved'), 'Gộp 5m làm mất số liệu'
data.session_gaps(df1, minutes=5, limit=15)

In [ ]:
df1.to_parquet(f'{OUT}/bars_1m.parquet', index=False)
df5.to_parquet(f'{OUT}/bars_5m.parquet', index=False)
json.dump({'audit_1m': audit, 'resample_checks': checks}, open(f'{OUT}/audit_report.json', 'w'), ensure_ascii=False, indent=1)
open(f'{OUT}/pip_freeze.txt', 'w').write(subprocess.run([sys.executable, '-m', 'pip', 'freeze'], capture_output=True, text=True).stdout)
print('Đã lưu:', os.listdir(OUT))